# Q1

(a) : None

(b) : '2026-05-06'

(c) : ['2026-05-06', '2026-05-18']

(d) : [('2026', '05', '06'), ('2026', '05', '18')]

(e) : ['2026-05-06', '2026-05-18']

추가질문 : (c)는 캡처그룹이 없어서 모든 문자열을 반환하고, (d)는 캡처크룹이 3개여서 그룹별로 묶인 리스트를 반환하고, (e)는 비캡처 그룹이어서 그룹 지정이 없는 것처럼 전체 매칭 문자열을 반환하기 때문이다. .

#Q2

(a) : '[T]!'

(b) : '[T]안녕[T] [T]세상[T]!'

(c) : '[T]안녕[T] [T]세상[T]!'

(d) : '수강생 <30>명, 조교 <3>명'

(e) : '수강생 <30>명, 조교 <3>명'

추가질문1 : (a)에서 .+는 탐욕적이기 때문에 가능한 긴 문자열을 매칭하지만, (b)에서 .+?는 게으르기 때문에 가장 짧게 매칭하기 때문이다.

추가질문 2 : 원시문자열에서는 \1이 그대로 첫번째 캡처 그룹으로 해석되기 때문이다.

# Q3

In [18]:
import re
from collections import Counter

_URL     = re.compile(r'https?://\S+')
_HTML    = re.compile(r'<[^>]+>')
_EMAIL   = re.compile(r'[\w.+-]+@[\w.-]+\.\w+')
_PHONE   = re.compile(r'\d{2,4}-\d{3,4}-\d{4}')
_MENTION = re.compile(r'@\w+')
_HASH    = re.compile(r'#\w+')
_JAMO    = re.compile(r'[\u3131-\u3163]+')
_SPACE   = re.compile(r'\s+')

_HASH_EXTRACT = re.compile(r'#([가-힣A-Za-z0-9]+)')

In [20]:
# (a)
def clean_post(post: str) -> str:
    post = _URL.sub(' ', post)
    post = _HTML.sub('', post)
    post, _ = _EMAIL.subn('[이메일]', post)
    post, _ = _PHONE.subn('[전화]', post)
    post = _MENTION.sub(' ', post)
    post = _HASH.sub(' ', post)
    post = _JAMO.sub('', post)
    post = _SPACE.sub(' ', post).strip()
    return post

In [21]:
# (b)
def extract_hashtags(post: str) -> list[str]:
    return _HASH_EXTRACT.findall(post)


In [22]:
# (c)
def analyze_posts(posts: list[str]) -> dict:
    cleaned = [clean_post(p) for p in posts]
    avg_len = round(sum(len(c) for c in cleaned) / len(cleaned), 2)

    all_tags = []
    for p in posts:
        all_tags.extend(extract_hashtags(p))
    counter = Counter(all_tags)
    hashtag_counts = dict(counter.most_common())

    masked_count = 0
    for p in posts:
        _, n1 = _EMAIL.subn('[이메일]', p)
        _, n2 = _PHONE.subn('[전화]', p)
        masked_count += n1 + n2

    return {
        "posts_n": len(posts),
        "avg_length_after_clean": avg_len,
        "hashtag_counts": hashtag_counts,
        "masked_count": masked_count,
    }

In [23]:
posts: list[str] = [" 오늘 #파이썬 수업 진짜 재밌었음!! @prof_kim @hong 감사 ㅎㅎ ",
                    " 자료: https://etl.snu.ac.kr/lec17 ",
                    " @lee @park 팀플 어디서 모이지ㅠㅠ #DCCP2026 #팀플 카톡 ㄱㄱ ",
                    " <b>중요</b>: 다음 시험 범위는 1-15강. " ,
                    " 문의는 mam3b@snu.ac.kr (010-1234-5678)로! " ,
                    " 여러 공백과\n\n\n줄바꿈이 많은 텍스트 " ,
                    " ㅋㅋㅋ #파이썬 진짜 좋다 #추천 https://snu.ac.kr ",
                    ]

In [24]:
for i in range(7):
  print(clean_post(posts[i]))

오늘 수업 진짜 재밌었음!! 감사
자료:
팀플 어디서 모이지 카톡
중요: 다음 시험 범위는 1-15강.
문의는 [이메일] ([전화])로!
여러 공백과 줄바꿈이 많은 텍스트
진짜 좋다


In [25]:
analyze_posts(posts)

{'posts_n': 7,
 'avg_length_after_clean': 13.57,
 'hashtag_counts': {'파이썬': 2, 'DCCP2026': 1, '팀플': 1, '추천': 1},
 'masked_count': 2}

설명 : 단계3, 단계4의 순서를 바꾸면, @가 먼저 매칭되어 이메일 패턴이 인식되지 않기 때문에 순서가 중요하다.